In [1]:
###############
from md_Helpers.run_logs import configure_run_logging

configure_run_logging(
    True,
    notebook_name="Otto_variable_Ncells",
)
###############

PosixPath('/exp/e961/data/MDsims-data/pnichols/run_logs/Otto_variable_Ncells_2026-08-05_20-58-27.log')

In [ ]:
from md_Helpers import hot_spike

densities = [0.720]
temperatures = [0.800]
energies = [600, 700, 800, 900, 1000]

radii = [3.0]
n_cells = [10, 20, 30, 40, 50, 60, 70, 80]
for n_cell in n_cells:
    for target_rho in densities:
        for kT in temperatures:
            for radius in radii:
                for injected_energy in energies:
                    result = hot_spike.get_or_create_hot_spike(
                        n_fcc_cells=n_cell,
                        target_rho=target_rho,
                        kT=kT,
                        source_nsteps=1_000_000,
                        source_seed=1,
    
                        radius=radius,
                        injected_energy=injected_energy,
                        method="velocity_rescale_com",
    
                        nsteps2=980_000,
                        dt2=0.005,
    
                        log_period=1_000,
                        trajectory_period=1_000,
    
                        random_location=False,
    
                        overwrite=False,
                        overwrite_initial=False,
                        overwrite_source=False,
                        create_source_if_missing=True,
                        reject_phase_separated_source=True,
                    )

Thermalized state exists: checking phase separation.
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_10/rho_0.720/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
Created new hot-spike initial state
state_path: /exp/e961/data/MDsims-data/pnichols/Excitation_States_v3/FCC/n_cells_10/source_kT_0.800/rho_0.720/source_nsteps_1000000/source_seed_1/method_velocity_rescale_com/radius_3.000/energy_600.000/center_box/excitation_initial.gsd
creation_metadata_path: /exp/e961/data/MDsims-data/pnichols/Excitation_States_v3/FCC/n_cells_10/source_kT_0.800/rho_0.720/source_nsteps_1000000/source_seed_1/method_velocity_rescale_com/radius_3.000/energy_600.000/center_box/excitation_creation.hdf5
method: velocity_rescale_com
radius: 3.0
requested_injected_energy: 600.0
actual_injected_energy: 600.0000000000001
selected_particle_count: 85
Using GPU device
Final device: <hoomd.device.GPU object at 0x7efd285f6cf0>
Starting Excitation segment 1 Evolution

In [ ]:
import matplotlib.pyplot as plt
from md_Helpers import master_csv

# Reads existing run metadata; it does not rerun the simulations.
df = master_csv.build_excitation_evolved_master_csv()

sweep = df[
    df["n_fcc_cells"].isin(n_cells)
    & df["energy_deposition"].isin(energies)
    & df["density"].isin(densities)
    & df["temp"].isin(temperatures)
    & df["radius"].isin(radii)
    & (df["excitation_method"] == "velocity_rescale_com")
].copy()

colors = {
    "phase_separated": "tab:blue",
    "not_phase_separated": "tab:orange",
}

labels = {
    "phase_separated": "Bubble / phase separated",
    "not_phase_separated": "Recondensed / no bubble",
}

fig, ax = plt.subplots(figsize=(8, 6))

for phase, group in sweep.groupby("voxel_phase"):
    ax.scatter(
        group["n_fcc_cells"],
        group["energy_deposition"],
        s=85,
        color=colors.get(phase, "gray"),
        label=labels.get(phase, phase),
        edgecolor="black",
        linewidth=0.5,
        zorder=3,
    )

ax.set(
    xlabel="Number of FCC cells",
    ylabel="Injected energy",
    title=r"Hot-spike outcome ($\rho=0.720$, $kT=0.800$, $r=3.0$)",
)

ax.set_xticks(n_cells)
ax.set_yticks(energies)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
display(
    sweep[
        ["n_fcc_cells", "energy_deposition", "voxel_phase"]
    ].sort_values(["n_fcc_cells", "energy_deposition"])
)

print(f"Found {len(sweep)} of {len(n_cells) * len(energies)} expected runs")